# CS 696 · Frame a document-reuse question

Define a unit of reuse, locate a source value and its governing passages, and state a claim about moving that value that could fail an audit.

Start with a problem familiar from a CS degree. Two records have different field names. In a database migration you might already know the intended correspondence. In regulatory document reuse, establishing that correspondence is part of the research problem. A reviewer needs to see which edition says what, where a value came from and what remains uncertain.

The teaching case uses an invented filing standard for lab instrument maintenance. A fictional publishing office issues Editions A and B. Every clause, person and filing below is synthetic. The old filing contains an item code, service day, technician note and signer name. The new edition changes how some of this information is recorded. Work entirely within the supplied excerpts.

## Choose one decision a reviewer could check

Take the question “Can we reuse this filing?” and narrow it to a single decision. Can the value Q17 in the old item-code field fill the new instrument-ID field unchanged? Identify the filing, edition, source component, target component and proposed operation. These details let a second person examine the very same claim. A conclusion about one component leaves the rest of the dossier open.

u = (filing, source edition, source field, target edition, target field, operation)

u denotes one proposed move. Here u = (F7, A, item_code, B, instrument_id, copy). The actual source value is F7[A, item_code] = Q17. F7 has four populated source fields, but u concerns exactly one of them.

In [1]:
clauses = {
    "A1": "Item code identifies the instrument.",
    "A2": "Service day is the calendar day when maintenance ended.",
    "A3": "Technician note may describe completed or planned work.",
    "A4": "Signer name identifies the person who signed the log.",
    "B1": "Copy the Edition A item code unchanged into instrument ID.",
    "B2": "Copy the Edition A service day unchanged into completion day.",
}
form = {
    "item_code": "Q17",
    "service_day": "2040-02-07",
    "technician_note": "Replace gasket.",
    "signer_name": "Signer P",
}

unit = ("F7", "A", "item_code", "B", "instrument_id", "copy")
filing, old, source, new, target, operation = unit
print(filing, old, source, repr(form[source]), new, target, operation)
print("populated source fields:", len(form))
assert form[source] == "Q17" and len(form) == 4

F7 A item_code 'Q17' B instrument_id copy
populated source fields: 4


**Discuss** · The instrument ID can be filled correctly. Does this establish that filing F7 can be submitted under the new edition?

The evidence concerns only one proposed field move. A whole-filing claim would also need a target-field inventory and evidence that each applicable requirement was met. Keep the decision at the granularity you actually checked.

## Trace a value through both editions

Read the two passages side by side. A1 defines the old item code as the instrument identifier. B1 explicitly instructs the filer to copy that old value unchanged. Record both clause IDs beside the value’s location in F7. Repeat the exercise with A2 and B2. A phrase about a date can describe a due date, a start date or a completion date. In this case the old clause identifies the end of maintenance and the new one names its source.

e(u) = (source-value location, source-definition clause, target-instruction clause)

e(u) records the evidence for a proposed move. For the completion day it is (F7.service_day, A2, B2). Reading F7 yields 2040-02-07. Reading A2 and B2 justifies preserving that day unchanged. The two moves therefore need two distinct source locations and four clause references.

In [2]:
clauses = {
    "A1": "Item code identifies the instrument.",
    "A2": "Service day is the calendar day when maintenance ended.",
    "A3": "Technician note may describe completed or planned work.",
    "A4": "Signer name identifies the person who signed the log.",
    "B1": "Copy the Edition A item code unchanged into instrument ID.",
    "B2": "Copy the Edition A service day unchanged into completion day.",
}
form = {
    "item_code": "Q17",
    "service_day": "2040-02-07",
    "technician_note": "Replace gasket.",
    "signer_name": "Signer P",
}
candidates = [
    ("item_code", "instrument_id", "A1", "B1"),
    ("service_day", "completion_day", "A2", "B2"),
]

for source, target, old_ref, new_ref in candidates:
    print(target, "<-", "F7." + source, repr(form[source]))
    for key in (old_ref, new_ref):
        print(key, clauses[key])
assert {old for _, _, old, _ in candidates} == {"A1", "A2"}

instrument_id <- F7.item_code 'Q17'
A1 Item code identifies the instrument.
B1 Copy the Edition A item code unchanged into instrument ID.
completion_day <- F7.service_day '2040-02-07'
A2 Service day is the calendar day when maintenance ended.
B2 Copy the Edition A service day unchanged into completion day.


**Discuss** · A reviewer has the text of A2 and B2 but cannot locate the day in F7. What part of the argument survives?

The reviewer can examine the field correspondence. They still lack evidence for a particular date in this filing. A valid correspondence and an available source value are separate premises.

## Try to break the reuse claim

Choose a modest, falsifiable claim for a prototype. For the two correspondences already read and approved by a person, it copies only if the old value and both clause texts are available. Deliberately remove B2. An implementation that still copies the completion day would violate the claim. Restoring B2 should restore that one move. Testing source availability this way is a first step. A program that finds a clause ID can still misunderstand its meaning.

ready(u) = has_value(u) ∧ has_source_clause(u) ∧ has_target_clause(u)

∧ means that all three checks must hold. The items being checked here are preapproved correspondence rows. On intact F7 both copies are ready. With B2 absent, one of two is ready and the completion-day row needs review.

In [3]:
clauses = {
    "A1": "Item code identifies the instrument.",
    "A2": "Service day is the calendar day when maintenance ended.",
    "A3": "Technician note may describe completed or planned work.",
    "A4": "Signer name identifies the person who signed the log.",
    "B1": "Copy the Edition A item code unchanged into instrument ID.",
    "B2": "Copy the Edition A service day unchanged into completion day.",
}
form = {
    "item_code": "Q17",
    "service_day": "2040-02-07",
    "technician_note": "Replace gasket.",
    "signer_name": "Signer P",
}
candidates = [
    ("item_code", "instrument_id", "A1", "B1"),
    ("service_day", "completion_day", "A2", "B2"),
]

def inspect(form, candidate, clauses):
    source, target, old_ref, new_ref = candidate
    value = form.get(source)
    if value is None or value == "":
        return (target, "review", "source value missing")
    if old_ref not in clauses or new_ref not in clauses:
        return (target, "review", "clause unavailable")
    return (target, "copy", value)

reduced = {key: text for key, text in clauses.items() if key != "B2"}
result = [inspect(form, row, reduced) for row in candidates]
for row in result:
    print(row)
assert [row[1] for row in result] == ["copy", "review"]

('instrument_id', 'copy', 'Q17')
('completion_day', 'review', 'clause unavailable')


**Discuss** · A developer replaces B2 with an unrelated passage but leaves the same dictionary key. Will this test catch the mistake?

No. The test checks whether the preapproved reference can be resolved. A reviewer must separately verify that the retrieved text authorizes the operation, including the edition and the meaning of the field.

## Independent task

Audit the two proposed copies under three conditions. Begin with the intact filing. In a second run withhold B2. In a third restore the clauses and remove only the item-code value. Predict which target can still be filled in each run. Produce a ledger row for each candidate with the source path, target field, clause IDs and decision.

In [4]:
clauses = {
    "A1": "Item code identifies the instrument.",
    "A2": "Service day is the calendar day when maintenance ended.",
    "A3": "Technician note may describe completed or planned work.",
    "A4": "Signer name identifies the person who signed the log.",
    "B1": "Copy the Edition A item code unchanged into instrument ID.",
    "B2": "Copy the Edition A service day unchanged into completion day.",
}
form = {
    "item_code": "Q17",
    "service_day": "2040-02-07",
    "technician_note": "Replace gasket.",
    "signer_name": "Signer P",
}
candidates = [
    ("item_code", "instrument_id", "A1", "B1"),
    ("service_day", "completion_day", "A2", "B2"),
]

runs = [
    ("intact", form, clauses),
    ("without B2", form, {k: v for k, v in clauses.items() if k != "B2"}),
    ("without item code", {k: v for k, v in form.items() if k != "item_code"}, clauses),
]
for name, record, available in runs:
    print(name, sorted(record), sorted(available))
# Add the ledger rows and compare them to your predictions.

intact ['item_code', 'service_day', 'signer_name', 'technician_note'] ['A1', 'A2', 'A3', 'A4', 'B1', 'B2']
without B2 ['item_code', 'service_day', 'signer_name', 'technician_note'] ['A1', 'A2', 'A3', 'A4', 'B1']
without item code ['service_day', 'signer_name', 'technician_note'] ['A1', 'A2', 'A3', 'A4', 'B1', 'B2']


## Compare your solution

Set aside the missing value and missing passage in separate runs. A missing passage should not change a move that cites a different passage.

In [5]:
clauses = {
    "A1": "Item code identifies the instrument.",
    "A2": "Service day is the calendar day when maintenance ended.",
    "A3": "Technician note may describe completed or planned work.",
    "A4": "Signer name identifies the person who signed the log.",
    "B1": "Copy the Edition A item code unchanged into instrument ID.",
    "B2": "Copy the Edition A service day unchanged into completion day.",
}
form = {
    "item_code": "Q17",
    "service_day": "2040-02-07",
    "technician_note": "Replace gasket.",
    "signer_name": "Signer P",
}
candidates = [
    ("item_code", "instrument_id", "A1", "B1"),
    ("service_day", "completion_day", "A2", "B2"),
]

def inspect(form, candidate, clauses):
    source, target, old_ref, new_ref = candidate
    value = form.get(source)
    if value is None or value == "":
        return (target, "review", "source value missing")
    if old_ref not in clauses or new_ref not in clauses:
        return (target, "review", "clause unavailable")
    return (target, "copy", value)

runs = [
    ("intact", form, clauses),
    ("without B2", form, {k: v for k, v in clauses.items() if k != "B2"}),
    ("without item code", {k: v for k, v in form.items() if k != "item_code"}, clauses),
]
outcomes = []
for name, record, available in runs:
    decisions = []
    for candidate in candidates:
        source, target, a, b = candidate
        _, decision, detail = inspect(record, candidate, available)
        print(name, "F7." + source, target, [a, b], decision, detail)
        decisions.append(decision)
    outcomes.append(decisions)
assert outcomes == [["copy", "copy"], ["copy", "review"], ["review", "copy"]]

intact F7.item_code instrument_id ['A1', 'B1'] copy Q17
intact F7.service_day completion_day ['A2', 'B2'] copy 2040-02-07
without B2 F7.item_code instrument_id ['A1', 'B1'] copy Q17
without B2 F7.service_day completion_day ['A2', 'B2'] review clause unavailable
without item code F7.item_code instrument_id ['A1', 'B1'] review source value missing
without item code F7.service_day completion_day ['A2', 'B2'] copy 2040-02-07


On the intact packet both target fields receive a value. With B2 absent, only instrument ID can be filled. With the old item code absent, only completion day can be filled. Both individual failures need a human follow-up, but one calls for a passage and the other for a value from the filing.

## Mentor review

Bring the four-field source inventory, one six-part decision unit, two passage pairs and the six audit rows from the exercise.

- Give a pinpoint source location for each value and preserve the edition and clause IDs for its proposed move.
- Show which claims survived each evidence-removal test and why.
- Record the exact reuse claim you and your mentor will pursue, plus one observation that would overturn it.
- What evidence would distinguish a renamed field from a field with a different meaning?
- Which parts of your hand audit could a program repeat, and where does someone still need to read the passage?

Give Module 02 your two confirmed copies with their evidence pointers. Carry forward the unresolved technician note and signer name. Those components will test what to do when the new edition splits an old field or requests additional metadata.